# Preprocessing — Eksperimen 1

Mengubah data mentah CamRest676 menjadi sample per-turn yang siap dilatih:
delexicalization, belief span, vocabulary, `DataLoader`, dan query knowledge base.
Setiap tahap menampilkan output agar perubahannya terlihat jelas.

## Konfigurasi

Konstanta pipeline: rasio split, special token, dan definisi slot.

In [2]:
import json, re, random
from pathlib import Path
from collections import Counter

import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# Cari root repo (folder yang memuat /data) dari lokasi notebook.
ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"

# Split 3:1:1 sesuai paper
TRAIN_RATIO, DEV_RATIO, TEST_RATIO = 0.6, 0.2, 0.2
SEED = 42

# Special tokens
PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN = "<pad>", "<sos>", "<eos>", "<unk>"
INF_OPEN, INF_CLOSE, REQ_OPEN, REQ_CLOSE = "<inf>", "</inf>", "<req>", "</req>"
SLOT_TOKENS = ["NAME_SLOT", "ADDRESS_SLOT", "PHONE_SLOT", "POSTCODE_SLOT",
               "FOOD_SLOT", "AREA_SLOT", "PRICERANGE_SLOT"]
SPECIAL_TOKENS = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN,
                  INF_OPEN, INF_CLOSE, REQ_OPEN, REQ_CLOSE] + SLOT_TOKENS

# Slot CamRest676
INFORMABLE_SLOTS = ["food", "area", "pricerange"]
DB_FIELD_TO_SLOT = {"name": "NAME_SLOT", "address": "ADDRESS_SLOT", "phone": "PHONE_SLOT",
                    "postcode": "POSTCODE_SLOT", "food": "FOOD_SLOT", "area": "AREA_SLOT",
                    "pricerange": "PRICERANGE_SLOT"}

print("Root repo:", ROOT)
print("Data dir :", DATA_DIR)
print(f"Special tokens ({len(SPECIAL_TOKENS)}):", SPECIAL_TOKENS)

Root repo: e:\coding bebas\restorant-asistent
Data dir : e:\coding bebas\restorant-asistent\data
Special tokens (15): ['<pad>', '<sos>', '<eos>', '<unk>', '<inf>', '</inf>', '<req>', '</req>', 'NAME_SLOT', 'ADDRESS_SLOT', 'PHONE_SLOT', 'POSTCODE_SLOT', 'FOOD_SLOT', 'AREA_SLOT', 'PRICERANGE_SLOT']


## 1. Load Data Mentah

Dua file: 676 dialog percakapan dan knowledge base 110 restoran.

In [3]:
def load_raw_data():
    with open(DATA_DIR / "CamRest676.json") as f:
        dialogues = json.load(f)
    with open(DATA_DIR / "CamRestDB.json") as f:
        database = json.load(f)
    return dialogues, database

dialogues, database = load_raw_data()

turn = dialogues[0]["dial"][0]
print(f"Total dialog : {len(dialogues)}")
print(f"Total KB     : {len(database)} restoran\n")
print("Contoh 1 turn mentah:")
print("  usr:", turn["usr"]["transcript"])
print("  sys:", turn["sys"]["sent"])
print("\nContoh entry KB:")
print(" ", database[0])

Total dialog : 676
Total KB     : 110 restoran

Contoh 1 turn mentah:
  usr: I need to find an expensive restauant that's in the south section of the city.
  sys: There are several restaurants in the south part of town that serve expensive food. Do you have a cuisine preference?

Contoh entry KB:
  {'address': 'Regent Street City Centre', 'area': 'centre', 'food': 'italian', 'location': '52.20103,0.126023', 'phone': '01223 323737', 'pricerange': 'cheap', 'postcode': 'C.B 2, 1 A.B', 'type': 'restaurant', 'id': '19210', 'name': 'pizza hut city centre'}


## 2. Split Data 3:1:1

Shuffle dengan `seed` tetap agar reproducible, lalu bagi train/dev/test.

In [4]:
def split_data(dialogues, seed=SEED):
    random.seed(seed)
    idx = list(range(len(dialogues)))
    random.shuffle(idx)
    n = len(dialogues)
    n_train, n_dev = int(n * TRAIN_RATIO), int(n * DEV_RATIO)
    train = [dialogues[i] for i in idx[:n_train]]
    dev = [dialogues[i] for i in idx[n_train:n_train + n_dev]]
    test = [dialogues[i] for i in idx[n_train + n_dev:]]
    return train, dev, test

train_dial, dev_dial, test_dial = split_data(dialogues)
print(f"Train: {len(train_dial)} | Dev: {len(dev_dial)} | Test: {len(test_dial)} dialog")

Train: 405 | Dev: 135 | Test: 136 dialog


## 3. Delexicalization

Ganti entitas milik restoran pada **response sistem** dengan placeholder
(`NAME_SLOT`, `FOOD_SLOT`, ...). Hanya response yang di-delex — input user & belief
span dibiarkan apa adanya.

In [5]:
def delexicalize_response(response, database):
    delex = response.lower()
    food, area, price = set(), set(), set()
    for entry in database:
        for field, slot in (("name", "NAME_SLOT"), ("address", "ADDRESS_SLOT"),
                            ("phone", "PHONE_SLOT"), ("postcode", "POSTCODE_SLOT")):
            val = entry.get(field, "").lower()
            if val and val in delex:
                delex = delex.replace(val, slot)
        if entry.get("food"): food.add(entry["food"].lower())
        if entry.get("area"): area.add(entry["area"].lower())
        if entry.get("pricerange"): price.add(entry["pricerange"].lower())
    # Nilai pendek & umum -> pakai word-boundary, terpanjang dulu.
    for val in sorted(food, key=len, reverse=True):
        delex = re.sub(rf"\b{re.escape(val)}\b", "FOOD_SLOT", delex)
    for val in sorted(area, key=len, reverse=True):
        delex = re.sub(rf"\b{re.escape(val)}\b", "AREA_SLOT", delex)
    for val in sorted(price, key=len, reverse=True):
        delex = re.sub(rf"\b{re.escape(val)}\b", "PRICERANGE_SLOT", delex)
    return delex

# Cari satu response yang benar-benar berubah, tampilkan sebelum/sesudah.
sample = next(t["sys"]["sent"] for d in train_dial for t in d["dial"]
              if delexicalize_response(t["sys"]["sent"], database) != t["sys"]["sent"].lower())
print("SEBELUM:", sample)
print("SESUDAH:", delexicalize_response(sample, database))

SEBELUM: Pizza Hut Cherry Hinton is in the south part of town and in the moderate price range.
SESUDAH: NAME_SLOT is in the AREA_SLOT part of town and in the PRICERANGE_SLOT price range.


## 4. Belief Span

Ringkasan kebutuhan user per turn dengan format
`<inf> value ; value </inf> <req> slot ; slot </req>`:
informable menyimpan **nilai** (mis. `italian`), requestable menyimpan **nama slot**
(mis. `address`).

In [6]:
def construct_bspan(slu_annotations):
    informable, requestable = [], []
    for slu in slu_annotations:
        act = slu["act"]
        for pair in slu["slots"]:
            if act == "inform":
                name, value = pair[0], pair[1]
                if value != "dontcare" and name in INFORMABLE_SLOTS and value.lower() not in informable:
                    informable.append(value.lower())
            elif act == "request":
                req = pair[1] if pair[0] == "slot" else pair[0]
                if req.lower() not in requestable:
                    requestable.append(req.lower())
    return (f"{INF_OPEN} {' ; '.join(informable)} {INF_CLOSE} "
            f"{REQ_OPEN} {' ; '.join(requestable)} {REQ_CLOSE}")

# Ambil turn yang punya act request agar informable & requestable sama-sama terlihat.
demo = next(t for d in train_dial for t in d["dial"]
            if any(s["act"] == "request" for s in t["usr"]["slu"]))
print("SLU  :", demo["usr"]["slu"])
print("Bspan:", construct_bspan(demo["usr"]["slu"]))

SLU  : [{'act': 'request', 'slots': [['slot', 'phone']]}, {'act': 'request', 'slots': [['slot', 'food']]}, {'act': 'request', 'slots': [['slot', 'address']]}, {'act': 'inform', 'slots': [['pricerange', 'moderate']]}, {'act': 'inform', 'slots': [['area', 'south']]}]
Bspan: <inf> moderate ; south </inf> <req> phone ; food ; address </req>


## 5. Format Sample per Turn

Susun pasangan input–target per turn (Eq. 4):
input `X = B_{t-1} R_{t-1} U_t`, target stage-1 `B_t`, target stage-2 `R_t`.

In [7]:
def process_dialogue(dialogue, database):
    processed, prev_bspan, prev_response = [], "", ""
    for t, turn in enumerate(dialogue["dial"]):
        user = turn["usr"]["transcript"].lower().strip()
        bspan = construct_bspan(turn["usr"]["slu"])
        response = delexicalize_response(turn["sys"]["sent"].lower().strip(), database)
        parts = [p for p in (prev_bspan, prev_response) if p] + [user]
        processed.append({
            "input": " ".join(parts),
            "target_bspan": bspan,
            "target_response": response,
            "dialogue_id": dialogue.get("dialogue_id", ""),
            "turn": t,
        })
        prev_bspan, prev_response = bspan, response
    return processed

def process_all(dialogues, database):
    return [s for d in dialogues for s in process_dialogue(d, database)]

train_samples = process_all(train_dial, database)
dev_samples = process_all(dev_dial, database)
test_samples = process_all(test_dial, database)

s = train_samples[3]
print("INPUT    :", s["input"])
print("BSPAN    :", s["target_bspan"])
print("RESPONSE :", s["target_response"])
print(f"\nTotal sample -> Train: {len(train_samples)}, Dev: {len(dev_samples)}, Test: {len(test_samples)}")

INPUT    : <inf> moderate ; south </inf> <req> phone ; food ; address </req> they serve FOOD_SLOT food and are located at ADDRESS_SLOT.  their number is PHONE_SLOT. thank you
BSPAN    : <inf> moderate ; south </inf> <req>  </req>
RESPONSE : you are very welcome. good bye.

Total sample -> Train: 1635, Dev: 553, Test: 556


## 6. Tokenisasi

Pisahkan tanda baca dari kata, lalu split berdasarkan spasi.

In [8]:
def tokenize(text):
    text = re.sub(r"([.,!?;:'\"\(\)])", r" \1 ", text)
    return re.sub(r"\s+", " ", text).strip().split()

print(tokenize(train_samples[3]["target_response"]))

['you', 'are', 'very', 'welcome', '.', 'good', 'bye', '.']


## 7. Vocabulary

Dibangun **hanya dari training set** (anti data leakage). Special token menempati
indeks awal, sisanya kata terurut.

In [9]:
def build_vocabulary(train_samples, min_freq=1):
    counter = Counter()
    for s in train_samples:
        counter.update(tokenize(s["input"]))
        counter.update(tokenize(s["target_bspan"]))
        counter.update(tokenize(s["target_response"]))
    word2idx = {tok: i for i, tok in enumerate(SPECIAL_TOKENS)}
    for word in sorted(w for w, c in counter.items() if c >= min_freq):
        word2idx.setdefault(word, len(word2idx))
    idx2word = {i: w for w, i in word2idx.items()}
    return word2idx, idx2word

def tokens_to_indices(tokens, word2idx):
    unk = word2idx[UNK_TOKEN]
    return [word2idx.get(t, unk) for t in tokens]

word2idx, idx2word = build_vocabulary(train_samples)
print("Vocab size:", len(word2idx))
print("12 entri pertama:", list(word2idx.items())[:12])

Vocab size: 755
12 entri pertama: [('<pad>', 0), ('<sos>', 1), ('<eos>', 2), ('<unk>', 3), ('<inf>', 4), ('</inf>', 5), ('<req>', 6), ('</req>', 7), ('NAME_SLOT', 8), ('ADDRESS_SLOT', 9), ('PHONE_SLOT', 10), ('POSTCODE_SLOT', 11)]


## 8. Dataset & DataLoader

Setiap sample diberi `<sos>`/`<eos>` untuk decoder, lalu di-pad per batch.
Token mentah (`input_tokens`, `bspan_tokens`) tetap disimpan karena dibutuhkan
mekanisme copy (CopyNet) di tahap training.

In [10]:
class SequicityDataset(Dataset):
    def __init__(self, samples, word2idx):
        self.samples = samples
        self.sos = word2idx[SOS_TOKEN]
        self.eos = word2idx[EOS_TOKEN]
        self.word2idx = word2idx

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        s = self.samples[i]
        input_tokens = tokenize(s["input"])
        bspan_tokens = tokenize(s["target_bspan"])
        resp_tokens = tokenize(s["target_response"])
        bspan_idx = tokens_to_indices(bspan_tokens, self.word2idx)
        resp_idx = tokens_to_indices(resp_tokens, self.word2idx)
        return {
            "input_indices": torch.tensor(tokens_to_indices(input_tokens, self.word2idx)),
            "input_tokens": input_tokens,
            "bspan_input": torch.tensor([self.sos] + bspan_idx),
            "bspan_target": torch.tensor(bspan_idx + [self.eos]),
            "bspan_tokens": bspan_tokens,
            "response_input": torch.tensor([self.sos] + resp_idx),
            "response_target": torch.tensor(resp_idx + [self.eos]),
        }

def collate_fn(batch):
    pad = lambda key: pad_sequence([b[key] for b in batch], batch_first=True, padding_value=0)
    return {
        "input_padded": pad("input_indices"),
        "input_lengths": torch.tensor([len(b["input_indices"]) for b in batch]),
        "input_tokens": [b["input_tokens"] for b in batch],
        "bspan_input": pad("bspan_input"),
        "bspan_target": pad("bspan_target"),
        "bspan_tokens": [b["bspan_tokens"] for b in batch],
        "response_input": pad("response_input"),
        "response_target": pad("response_target"),
    }

def get_dataloader(samples, word2idx, batch_size, shuffle=True):
    return DataLoader(SequicityDataset(samples, word2idx), batch_size=batch_size,
                      shuffle=shuffle, collate_fn=collate_fn)

batch = next(iter(get_dataloader(train_samples, word2idx, 4, shuffle=False)))
print("Shape tensor dalam 1 batch (batch_size=4):")
for k, v in batch.items():
    if torch.is_tensor(v):
        print(f"  {k:15s}: {tuple(v.shape)}")

Shape tensor dalam 1 batch (batch_size=4):
  input_padded   : (4, 43)
  input_lengths  : (4,)
  bspan_input    : (4, 13)
  bspan_target   : (4, 13)
  response_input : (4, 19)
  response_target: (4, 19)


In [11]:
# Verifikasi: decode kembali 1 sample dari batch ke teks.
def decode(indices):
    return " ".join(idx2word[i.item()] for i in indices if idx2word[i.item()] != PAD_TOKEN)

print("input    :", decode(batch["input_padded"][0]))
print("bspan    :", decode(batch["bspan_target"][0]))
print("response :", decode(batch["response_target"][0]))

input    : i want a restaurant that is moderately priced and located in the south .
bspan    : <inf> moderate ; south </inf> <req> </req> <eos>
response : NAME_SLOT is in the AREA_SLOT part of town and in the PRICERANGE_SLOT price range . <eos>


## 9. Knowledge Base

Belief span dipakai untuk query KB: parse constraint informable, cari restoran yang
cocok, hasilkan indikator `kt` = `[no_match, exact_match, multiple_match]`.
Saat inference, placeholder diganti balik ke nilai asli (lexicalize).

In [12]:
def parse_bspan(text):
    inf = re.search(r"<inf>\s*(.*?)\s*</inf>", text)
    req = re.search(r"<req>\s*(.*?)\s*</req>", text)
    informable = [v.strip() for v in inf.group(1).split(";") if v.strip()] if inf and inf.group(1).strip() else []
    requestable = [v.strip() for v in req.group(1).split(";") if v.strip()] if req and req.group(1).strip() else []
    return informable, requestable

def search_kb(bspan_text, database):
    informable, _ = parse_bspan(bspan_text)
    if not informable:
        return database, torch.tensor([0., 0., 1.])
    matches = [e for e in database
               if all(any(e.get(slot, "").lower() == v.lower() for slot in INFORMABLE_SLOTS)
                      for v in informable)]
    n = len(matches)
    kt = torch.tensor([1., 0., 0.]) if n == 0 else torch.tensor([0., 1., 0.]) if n == 1 else torch.tensor([0., 0., 1.])
    return matches, kt

def lexicalize_response(response, kb_matches):
    if not kb_matches:
        return response
    entry = kb_matches[0]
    for field, slot in DB_FIELD_TO_SLOT.items():
        val = entry.get(field, "")
        if val:
            response = response.replace(slot.lower(), val.lower()).replace(slot, val)
    return response

# Demo pakai bspan asli yang punya constraint.
bspan = next(s["target_bspan"] for s in train_samples if parse_bspan(s["target_bspan"])[0])
informable, requestable = parse_bspan(bspan)
matches, kt = search_kb(bspan, database)

print("Bspan       :", bspan)
print("Informable  :", informable)
print("Requestable :", requestable)
print(f"KB matches  : {len(matches)} restoran | kt = {kt.tolist()}")
if matches:
    m = matches[0]
    print(f"Match #1    : {m['name']} ({m.get('food')}, {m.get('area')}, {m.get('pricerange')})")
    print("Lexicalize  :", lexicalize_response("NAME_SLOT is located at ADDRESS_SLOT .", matches))

Bspan       : <inf> moderate ; south </inf> <req>  </req>
Informable  : ['moderate', 'south']
Requestable : []
KB matches  : 2 restoran | kt = [0.0, 0.0, 1.0]
Match #1    : pizza hut cherry hinton (italian, south, moderate)
Lexicalize  : pizza hut cherry hinton is located at G4 Cambridge Leisure Park Clifton Way Cherry Hinton .
